# Optimisation des hyperparamètres des filtres 
### On explore ici une méthode d'optimisation des paramètres des filtres linéaires pour avoir un seuillage efficace sur les résidus des paramètres de TLE filtrés. 
Script final écrit dans optimise_seuil.py

In [ ]:
from optimisation_seuil.metrics import minimize 
import numpy as np
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt


from metrics import confusion_matrix, precision_recall_f1, lissage_noyau_gaussien_metriques
from optimise_seuil import optimise_seuil_kalman_via_regul_gaussienne, to_days
from maneuver_detection.discrete_kalman_filter import detect_kalman
def find_root(marker: str = "pyproject.toml"):
    p = Path.cwd().resolve()
    for parent in (p, *p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"{marker} introuvable en remontant depuis {p}")

ROOT = find_root()
FIG_DIR = ROOT / "data" / "graphs" 


On va ici essayer d'optimiser cette méthode pour les 4 satellites dont on connait l'historique des manoeuvres vraies.

On introduit la fonction suivante qui mesure la performance du filtre :

$$
F_1 = \dfrac{2 \text{Pr}  * \text{Re}}{\text{Pr}  + \text{Re}}
$$

où 

$$
\text{Pr} = \dfrac{n_{det}}{n_{det} + n_{false}} \quad \text{Re} = \dfrac{n_{det}}{n_{det} + n_{missed}}
$$
Le but est d'optimiser cette métrique en fonction des paramères $r, var_Q, P0$ du filtre Kalman d'ordre 1 ou 2.
Problème : cette métrique est non régulière (constante par morceaux). 
On va donc la régulariser via un produit de convolution avec un noyau gaussien.

Supposons donc que l'on dispose (en jours % $t_0$) de $t_i$ des dates vraies de manoeuvres et $\tilde{t_i}$ des dates prédites de manoeuvres par filtrage des résidus du paramètre considéré (demi-grand axe, inclinaison, perigé, ...), ainsi que d'un score $s_i$ de la détection. Par exemple 
$$ 
s_i = | \varepsilon (\tilde{t_i}) - \text{seuil} | 
$$ 
Avec $\text{seuil} = quantile(\chi ^2(1), \alpha)$. 
On rappelle que $ \varepsilon \sim \chi ^ 2 (1) $ en l'absence de manoeuvres.

On introduit alors $ \sigma$ et $\beta$, deux paramètres de lissage.

$$
w_{ij} = e^{-\dfrac{(t_i - \tilde{t_j})^2}{2\sigma ^2}} \in \mathbb{R}^{n*n}
$$
matrice de poids de correspondance temporelle.
$$
p_j = \text{sigmoid}(\dfrac{s_j - seuil}{\beta}) \in \mathbb{R}^n 
$$
vecteur de probabilité de détection. 

Ainsi, les métriques lissées deviennent : 
$$
\tilde{\text{Pr}} = \dfrac{1}{n} \sum_{i = 1}^n \text{max}_j(w_{ij} p_j) \quad \tilde{\text{Re}} = \sum_{i = 1}^n p_i \text{max}_j(w_{ij})
$$
Puis : 
$$
\tilde{F_1}(X) = \dfrac{2 \tilde{\text{Pr}}  * \tilde{\text{Re}}}{\tilde{\text{Pr}}  + \tilde{\text{Re}}} \in \mathcal{C}^1 (\mathbb{R}^d, \mathbb{R})
$$
avec $X$ le vecteur des paramètres

In [2]:
# on plot d'abord l'allure de F1 et F1_lissée en fonction de r. 

def plot_f1_r_direction(path_tle, path_man, norad, r_values, var_Q=0.05, p0=1e+4, sigma_lissage=0.5, beta_lissage=0.5, tol_days=1.0, alpha=0.997,
            save_path=None):

    df = (
        pl.scan_parquet(path_tle).filter(pl.col('norad') == norad)
        .select('epoch', 'sma', 'mean_motion', 'raan', 'arg_perigee', 'mean_anomaly', 'inclination', 'eccentricity')
        .sort('epoch').collect()
        )

    true_maneuvers = pl.scan_parquet(path_man).collect()
    true_maneuvers_dates = true_maneuvers["burn_epoch"].to_numpy()
    e = df["epoch"].to_numpy()[1:]

    f1_scores = []
    f1_lisse_scores = []

    for r in r_values:
        epoch, sma_dot, nis, predictions, threshold = detect_kalman(df, ordre=1, var_Q=var_Q,r=r, p0=p0, alpha=alpha, plot=False)
        t0 = epoch[0]

        ## manoeuvres vraies t_i, restreintes à la fenêtre couverte par les données, en jours depuis t0
        true_dt = true_maneuvers_dates[(true_maneuvers_dates >= e[0]) & (true_maneuvers_dates <= e[-1])]
        t_i = to_days(true_dt, t0)

        ## F1 classique : seuillage dur, `predictions` = indices des epochs détectées comme manoeuvres
        t_i_tilde = to_days(epoch[predictions], t0)
        conf_mat = confusion_matrix(t_i, t_i_tilde, tol_days=tol_days)
        f1 = precision_recall_f1(conf_mat)["f1"]
        
        ## F1 lissée : tous les candidats + score nis, pondération gaussienne/sigmoïde
        epoch_d = to_days(epoch, t0)
        f1_lisse = 1.0 - lissage_noyau_gaussien_metriques(
            t_i, epoch_d, nis, threshold, sigma_lissage, beta_lissage
        )

        f1_scores.append(f1)
        f1_lisse_scores.append(f1_lisse)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(r_values, f1_scores, ms=3, label="F1")
    ax.plot(r_values, f1_lisse_scores, ms=3, label="F1 lissée")
    ax.set_xscale("log")
    ax.set_xlabel("r")
    ax.set_ylabel("F1")
    ax.set_title(f"F1 vs F1 lissée en fonction de r (norad={norad})")
    ax.legend()
    plt.tight_layout()
    
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()

    return f1_scores, f1_lisse_scores

In [3]:
r_values = np.logspace(-8, 1, 2000)
#  sentinel 3b
path_man = ROOT / 'data' / 'raw' /  "s3b_man_ILRS.parquet"
path_tle = ROOT / 'data' / 'raw' /  "SENTINEL_ALL_1783007466.119748.parquet"
norad = 43437

save_path= FIG_DIR / f"{norad}_f1_vs_r.png"
#f1_scores, f1_lisse_scores = plot_f1_r_direction(path_tle, path_man, norad, r_values=r_values, sigma_lissage=0.9, beta_lissage=1.5, save_path= save_path, tol_days=1)


Les paramètres de lissage $\sigma = 0.9, \; \beta = 1.5$ donnent un bon résultat : la fonction est bien régularisée et le max global est conservé. 

Ici à vue d'oeil, le max est atteint pour $r = 8.10^{-5}$ km. On conserve pour l'instant cette valeur pour ce satellite et on teste selon l'axe $varQ$.

In [4]:
# on plot l'allure de F1 et F1_lissée en fonction de varQ. on se base sur sentinel 3b

def plot_f1_varQ_direction(varQ_values, r = 8e-5, p0=1e+4, sigma_lissage=0.5, beta_lissage=0.5, tol_days=1.0, alpha=0.997,
            save_path=None):
    df = (
        pl.scan_parquet(path_tle).filter(pl.col('norad') == norad)
        .select('epoch', 'sma', 'mean_motion', 'raan', 'arg_perigee', 'mean_anomaly', 'inclination', 'eccentricity')
        .sort('epoch').collect()
        )

    true_maneuvers = pl.scan_parquet(path_man).collect()
    true_maneuvers_dates = true_maneuvers["burn_epoch"].to_numpy()
    e = df["epoch"].to_numpy()[1:]

    f1_scores = []
    f1_lisse_scores = []

    for var_Q in varQ_values:
        epoch, sma_dot, nis, predictions, threshold = detect_kalman(
            df, ordre=1, r=r, var_Q=var_Q, p0=p0, alpha=alpha, plot=False
        )
        t0 = epoch[0]
        
        ## manoeuvres vraies t_i, restreintes à la fenêtre couverte par les données, en jours depuis t0
        true_dt = true_maneuvers_dates[(true_maneuvers_dates >= e[0]) & (true_maneuvers_dates <= e[-1])]
        t_i = to_days(true_dt, t0)

        ## F1 classique : seuillage dur, `predictions` = indices des epochs détectées comme manoeuvres
        t_i_tilde = to_days(epoch[predictions], t0)
        conf_mat = confusion_matrix(t_i, t_i_tilde, tol_days=tol_days)
        f1 = precision_recall_f1(conf_mat)["f1"]

        ## F1 lissée : tous les candidats + score nis, pondération gaussienne/sigmoïde
        epoch_d = to_days(epoch, t0)
        f1_lisse = 1.0 - lissage_noyau_gaussien_metriques(
            t_i, epoch_d, nis, threshold, sigma_lissage, beta_lissage
        )

        f1_scores.append(f1)
        f1_lisse_scores.append(f1_lisse)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(varQ_values, f1_scores, ms=3, label="F1")
    ax.plot(varQ_values, f1_lisse_scores, ms=3, label="F1 lissée")
    ax.set_xscale("log")
    ax.set_xlabel("varQ")
    ax.set_ylabel("F1")
    ax.set_title(f"F1 vs F1 lissée en fonction de varQ (norad={norad})")
    ax.legend()
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()

    return f1_scores, f1_lisse_scores

In [5]:

def test_optimise(path_tle, path_man, norad, r_0, varQ_0, p0_0, sigma, beta, tol_days):
    optimise_seuil_kalman_via_regul_gaussienne(path_tle, path_man,norad, r_0=r_0, varQ_0 = varQ_0, p0_0=p0_0,sigma_lissage=sigma, beta_lissage=beta, tol_days=tol_days)
    

In [6]:
varQ_values = np.logspace(-8, 4, 2000)
save_path= FIG_DIR / f"{norad}_f1_vs_varQ.png"
#f1_scores, f1_lisse_scores = plot_f1_varQ_direction(varQ_values=varQ_values, sigma_lissage=0.9, beta_lissage=1.5, save_path= save_path, tol_days=1)


On observe un max pour $varQ = 10^{-2}$ à vue d'oeil. Regardons maintenant ce qu'on obtient avec un algorithme rigoureux de recherche d'extremum sur la fonction $F1$ lissée.

In [7]:
## Pour Sentinel-3B 43437
path_man = ROOT / 'data' / 'raw' /  "s3b_man_ILRS.parquet"
path_tle = ROOT / 'data' / 'raw' /  "SENTINEL_ALL_1783007466.119748.parquet"

#test_optimise(path_tle, path_man, r_0 = 8e-5, varQ_0 = 10e-2, p0_0=1000, norad=43437, sigma=0.5, beta=0.5, tol_days=1 )

On obtient des valeurs cohérentes avec ce qu'on obtenait en tatonnant variable par variable.

Testons sur les 3 autres satellites  

In [12]:
## Pour envisat 27386
path_man = ROOT / 'data' / 'raw' /  "en1_man_ILRS.parquet"
path_tle = ROOT / 'data' / 'raw' / "ENVISAT_ALL_1783068650.78004.parquet"
norad = 27386
test_optimise(path_tle, path_man, r_0 = 0.05, varQ_0 = 0.1, p0_0=1000, norad=27386, sigma=0.5, beta=0.5, tol_days=1)

r_values = np.logspace(-8, 1, 1000)
save_path= FIG_DIR / f"{norad}_f1_vs_r.png"

#f1_scores, f1_lisse_scores = plot_f1_r_direction(path_tle, path_man, norad, r_values=r_values, sigma_lissage=0.5, beta_lissage=0.5, save_path= save_path, tol_days=1)


norad=27386  var_Q=1.991e-06 r=0.05008 p0=259.3 F1=0.120 P=0.088 R=0.186  {'n_miss': 57, 'n_false': 134, 'n_det': 13}


In [13]:
## Pour cryosat2 36508
path_man = ROOT / 'data' / 'raw' /  "cs2_man_ILRS.parquet"
path_tle = ROOT / 'data' /'raw' / 'CRYOSAT_ESA_1783006097.0149062.parquet'

r_values = np.logspace(-8, 1, 2000)
save_path= FIG_DIR / "36598_f1_vs_r.png"

#f1_scores, f1_lisse_scores= plot_f1_r_direction(path_tle, path_man, 36508, r_values=r_values, sigma_lissage=0.9, beta_lissage=1.5, save_path=save_path, tol_days=1)

#test_optimise(path_tle, path_man, r_0 = 0.05, varQ_0 = 0.1, p0_0=1000, norad=36508, sigma=0.9, beta=1.5, tol_days=1)

In [14]:
## Pour saral 39086
path_man = ROOT / 'data' / 'raw' /  "srl_man_ILRS.parquet"
path_tle = ROOT / 'data' /'raw' / 'SARAL_ALL_1783069319.365695.parquet'
norad = 39086
r_values = np.logspace(-8, 1, 1000)
save_path= FIG_DIR / f"{norad}_f1_vs_r.png"

#f1_scores, f1_lisse_scores= plot_f1_r_direction(path_tle, path_man, norad , r_values=r_values, sigma_lissage=0.9, beta_lissage=1.5, save_path=save_path, tol_days=1)

#test_optimise(path_tle, path_man, r_0 = 0.05, varQ_0 = 0.1, p0_0=1000, norad=norad, sigma=0.9, beta=1.5, tol_days=1)

In [15]:
# On va stocker ces valeurs optimisées des paramètres. 
# norad=43437  var_Q=0.01086 r=3.321e-06 p0=6.192e+04 F1=0.438 P=0.400 R=0.485  {'n_miss': 17, 'n_false': 24, 'n_det': 16}
# norad=27386  var_Q=1.991e-06 r=0.05008 p0=259.3 F1=0.120 P=0.088 R=0.186  {'n_miss': 57, 'n_false': 134, 'n_det': 13}
# norad=36508  var_Q=0.0004361 r=2.859e-05 p0=1 F1=0.680 P=0.680 R=0.680  {'n_miss': 8, 'n_false': 8, 'n_det': 17}
# norad=39086  var_Q=4.842e-06 r=0.0002417 p0=96.99 F1=0.600 P=0.429 R=1.000  {'n_miss': 0, 'n_false': 16, 'n_det': 12}
params_LKF = {
    43437 : { 'r' : 3.321e-6, 
             'varQ' : 0.01086,
             'p0' : 6.192e+4
            },
    27386 : { 'r' : 0.05008, 
             'varQ' : 1.99e-6,
             'p0' : 259.3
            },
    36508 : { 'r' : 2.859e-5, 
             'varQ' : 4.3e-4,
             'p0' : 1
            },
    39086 : { 'r' : 2.41e-4, 
             'varQ' : 4.842e-6,
             'p0' : 96.99
            }
        }       